# <strong>TCP SYN flood</strong>

This exercise demonstrates a well-known denial-of-service attack, called <strong>TCP SYN flood</strong>. Students will be able to create a real attack using SPHERE tools, and to observe its effect on legitimate traffic. Afterwards, they will be asked to apply a known defense against SYN flood known as <strong>SYN cookies</strong>, repeat the attack and observe the protection.

This exercise helps students learn the following concepts: (1) How TCP/IP works and how its design can be misused for attacks, (2) How easy it is to perpetrate a DoS attack, with fully legitimate traffic and at a low rate, (3) How easy it is to protect machines from this type of attacks via built-in OS mechanisms. Additionally, extra credit questions improve a student's understanding of how networks and TCP/IP work. 

<strong>This lab will contain four topics:</strong>

1. Generating Legitimate Traffic
2. Turning Off SYN Cookies
3. Generating Attack Traffic
4. Collecting Statistics

### Step 0: Starting the Lab

Click the button to begin creating the experiment.

<strong>Note:</strong> If your buttons are not displaying, click on the <img width='20px' height='20px' style='margin-left: 1px;' src='resources/fast_forward.png'> icon at the top of your notebook to render all widgets.

In [3]:
# Click the button below to start your lab.
import os
import subprocess
import re
import threading
import queue
import time
import sys
from IPython.display import display, HTML
import ipywidgets as widgets
import logging

# This is a troublesome file that will throw unnecessary warnings and other errors.
# Delete it if it exists. (Wasn't an issue in older Jupyter versions.)
# !(rm -f ~/.local/share/jupyter/nbsignatures.db)

# Global variable for the checker script.
runAllSteps = False

# Adding the "resource/" directory so that we can import the start.py file.
module_dir = os.path.join(os.getcwd(), 'resources')
if module_dir not in sys.path:
    sys.path.append(module_dir)

# Importing the prepare_lab function.
from functions import *

# Defining some stuff for the output below.
output0 = widgets.Output()
startButton = widgets.Button(description="Start Lab")
labname = "synflood"

# Defining the button handler for startButton.
def on_start_clicked(b):
    prepare_lab(labname, output0)

# Providing on_click functionality.
startButton.on_click(on_start_clicked)

# Display button and output area.
display(startButton, output0)

Button(description='Start Lab', style=ButtonStyle())

Output()

<hr>

If you previously stopped your lab, you may restore your progress below by clicking "Load Lab". <u>You do not have to load your lab if you signed out, closed your notebook, or exited your node(s) or XDC by using ```exit```.</u>

In [3]:
# Click the button below to load your lab.
def loadlab(b):
    load_lab(labname, output0_2)

# Creating the button.
loadButton = widgets.Button(description="Load Lab")

# Creating an output area.
output0_2 = widgets.Output()

# Run the command on click.
loadButton.on_click(loadlab)

# Display the output.
display(loadButton, output0_2)

Button(description='Load Lab', style=ButtonStyle())

Output()

## <strong>Introduction</strong>

All students should have completed an introductory networking course with grade B or better.

- <a href="http://en.wikipedia.org/wiki/SYN_flood">Short summary of SYN flood attack on Wikipedia</a>
- SYN flood attacks in the <a href="http://www.amazon.com/Internet-Denial-Service-Mechanisms-Networking/dp/0131475738/ref=sr_1_1?ie=UTF8&s=books&qid=1212642071&sr=8-1">Internet Denial of Service</a> book (optional reading)
- <a href="http://cr.yp.to/syncookies.html">SYN cookie overview</a>
- <a href="http://www.tcpdump.org/tcpdump_man.html">Tcpdump's man page</a>

Denial of service attacks deny service to legitimate clients by tying up resources at the server with a flood of legiitmate-looking service requests or junk traffic. Before proceeding to the assignment instructions make sure that you understand how TCP SYN flood attack works, which resource it ties up and how, and how syncookies help mitigate this attack. 

Upon starting your lab, you will have a topology that looks like this.

<div style="text-align: center; padding-right: 40px"><img src="resources/synflood/topo.png"></div>

### Step 1: Create a Web Traffic Stream

You will start this lab by simulating network traffic. This can be easily done by creating a Bash script and using the `curl` command. If you are unfamiliar with `curl`, you may view the tutorial <a href="https://curl.se/docs/tutorial.html">here</a>.

<strong>Your task:</strong> SSH onto the `client` node by typing `ssh client`. Inside of your home directory (`/home/USERNAME_GOES_HERE`), create a file named `stream.sh`. You will need to include a shebang, which will allow you to call the script from the command line. A sample script will look like this:

```
#!/bin/bash

(Your Solution Here)
```

Inside of `(Your Solution Here)`, create a `curl` call that gets `index.html` every second. Consider using a <a href="https://www.warp.dev/terminus/bash-while-loop">while true</a> loop, and using <a href="https://en.wikipedia.org/wiki/Sleep_(command)">sleep</a>.

Once you have a working script, click on the button below to check your work. The notebook will test your script, and ensure that you have `curl` within your script.

In [4]:
# Click the button below to check your work.
step1Complete = False

# Function to check the permissions.
def step_1():
    # Important variables that must be accessed outside of this function.
    global step1Complete, result

    with output1:
        output1.clear_output()
        display(HTML("<span><img width='14px' height='14px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    # This subprocess statement is a little different. Need to initiate environment variables at the same time when running the command.
    result = subprocess.run('ssh -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@buffer /home/.checker/section_1.py 1 NA', shell=True, capture_output=True, text=True)
    
    if (result.returncode == 0):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: green;'>Success! You may continue onto the next step.</span>"))
            step1Complete = True

    elif (result.returncode == 1):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>Your C file compiles, but it does not print your username (USERNAME_GOES_HERE). Try again.</span>"))
            step1Complete = False
    
    elif (result.returncode == 2):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>A file named '/home/USERNAME_GOES_HERE/topic_1/step_1.c' cannot be found. Ensure that this file exists and is located within topic_1/.</span>"))
            step1Complete = False

    elif (result.returncode == 3):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'>Compiling your file resulted in an error. Somewhere in your file, your C syntax appears incorrect. Check again. Did you forget a semicolon?</span>"))
            step1Complete = False

def check_step_1(b):
    if (warn_student(labname)):
        output1.clear_output()
        with output1:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_1()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "1", result.returncode)

# Creating the button.
button = widgets.Button(description="Check File")

# Creating an output area.
output1 = widgets.Output()

# Run the command on click.
button.on_click(check_step_1)

# Display the output.
display(button, output1)

Button(description='Check File', style=ButtonStyle())

Output()

### Step 2: Disabling SYN Cookies

In this second question, you are going to experiment how to tinker with SYN cookies in your Linux environment.

SYN cookies are often on by default in Linux and FreeBSD. To check if they are on, type the following on `server` node: `sudo sysctl net.ipv4.tcp_syncookies`

You should see `1` as the result. 

<strong>Your task:</strong> SYN cookies must be set to zero for the demo to work. Type the following two commands on the `server` node:
```
sudo sysctl -w net.ipv4.tcp_syncookies=0
sudo sysctl -w net.ipv4.tcp_max_syn_backlog=10000
```

Verify that SYN cookies are turned off, then click on the button below to check your work.

In [2]:
# Click the button below to check your work.

# Function to check if the file was created and perms were changed.
def step_2():
    # Important variables that must be accessed outside of this function.
    global step2Complete, step3Complete, result

    # Loading, in case the check is slow.
    with output2:
        output2.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run('ssh -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@intro /home/.checker/step2.py', shell=True)

    # Note: If already completed, it cannot be accidentally "undone" since it's part of the next step.
    if (result.returncode == 0 or step3Complete) or step2Complete:
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: green;'>Success! You may continue onto the next step.</span>"))
            step2Complete = True
            
    elif (result.returncode == 2):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'>The text file is created, but you forgot to change the permissions.</span>"))
            step2Complete = False
            
    else:
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'>Check your work and try again.</span>"))
            step2Complete = False

def check_step_2(b):
    if (warn_student(labname)):
        output2.clear_output()
        with output2:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_2()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "2", result.returncode)

# Creating the button.
button = widgets.Button(description="Check File")

# Creating an output area.
output2 = widgets.Output()

# Run the command on click.
button.on_click(check_step_2)

# Display the output.
display(button, output2)

Button(description='Check File', style=ButtonStyle())

Output()

### Step 3: The Flooder Tool

Now, you will create your first SYN flood attack by using the Flooder tool.

<strong>Your task:</strong> Create a SYN flood between the `attacker` and the `server` nodes, using the Flooder tool. You can type `flooder` on the `attacker` node's command line to get a man page for the tool. 

For example: `flooder --dst server --src 1.2.0.0 --srcmask 255.255.0.0 --highrate 100 --proto 6` will send a flood of 100 SYN packets per second to the target called `server`, spoofing addresses from 1.2.0.0/16 range. You should make sure to spoof within <strong>1.1.2.0</strong> range (use mask <strong>255.255.255.0</strong>). 

Most flooder commands require a "sudo" in front. 

Once you have a functional command, type it into the text entry below. Your command will be tested, and ensure that it creates a SYN flood.

In [ ]:
# Click the button below to check your work.

# Function to check if the folder exists
def step_3():
    # Important variables that must be accessed outside of this function.
    global step1Complete, step3Complete, result

    # Loading, in case the check is slow.
    with output3:
        output3.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run(['ssh -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@intro /home/.checker/step3.py /home/USERNAME_GOES_HERE/jupyterintro'], shell=True)
    
    if (result.returncode == 0 and step1Complete):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: green;'>Success! You may continue onto the next step.</span>"))
            step3Complete = True
    elif step1Complete == False:
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'>You need to complete Step 1 before continuing.</span>"))
            step3Complete = False
    else:
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'>Check your work and try again.</span>"))
            step3Complete = False

def check_step_3(b):
    if (warn_student(labname)):
        output3.clear_output()
        with output3:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_3()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "3", result.returncode)

# Creating the button.
button = widgets.Button(description="Check Folder")

# Creating an output area.
output3 = widgets.Output()

# Run the command on click.
button.on_click(check_step_3)

# Display the output.
display(button, output3)

### Step 4: Collecting Statistics (Part 1)

You will now collect `tcpdump` statistics on client machine with and without syncookies. Then, you will calculate connection duration and draw graphs of connection duration on y-axis and connection start time on x-axis.

<strong>Before you start the next task:</strong> Stop all traffic by stopping your legitimate client's script and flooder if you haven't done so already. Navigate to the `client` and start a `tcpdump` with the following command:
```
ip route get 5.6.7.8
```

You should see something like this as a result: 
```
5.6.7.8 via 1.1.2.2 dev eth2  src 1.1.2.3
   cache mtu 1500 advmss 1460 metric 10 64

```

Thus the interface name leading to `5.6.7.8` is <strong>eth2</strong>. To see the traffic flowing, type: 
```
sudo tcpdump -nn -i eth2 
```

then generate some traffic and restart your legitimate client code.

<strong>Your task:</strong> You will need to discover proper `tcpdump` options to see <em>specifically</em> IP traffic. Save the recorded traffic into a file named `traffic.txt`. Store this in your home directory (`/home/USERNAME_GOES_HERE`). 

Make sure that you start `tcpdump` with the options provided above. 

The notebook will check to make sure that you have traffic that looks like the following:
(Adding soon).

In [ ]:
# Click the button below to check your work.
step4Complete = False

# Function to check if the file was deleted.
def step_4():
    # Important variables that must be accessed outside of this function.
    global step4Complete, result

    # Loading, in case the check is slow.
    with output4:
        output4.clear_output()
        display(HTML("<span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    result = subprocess.run(['ssh -i /home/USERNAME_GOES_HERE/.ssh/merge_key USERNAME_GOES_HERE@intro /home/.checker/step4'], shell=True)

    if (result.returncode == 0):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: green;'>Success! You may continue onto the next step.</span>"))
            step4Complete = True
            
    elif (result.returncode == 1):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'>Check your work and try again.</span>"))
            step4Complete = False
    

def check_step_4(b):
    if (warn_student(labname)):
        output4.clear_output()
        with output4:
            display(HTML("<span style='color: red;'><strong>WARNING:</strong> You have an autosaved lab that you have not yet loaded. If you would like to load your progress, click \"Load Lab\" at the top of the notebook. Otherwise, clicking on this button again will assume you're restarting the lab!</span>"))
    else:
        step_4()

        # Auto-save.
        if (not runAllSteps):
            trigger_save(labname, "4", result.returncode)

# Creating the button.
button = widgets.Button(description="Check File")

# Creating an output area.
output4 = widgets.Output()

# Run the command on click.
button.on_click(check_step_4)

# Display the output.
display(button, output4)

### Step 5: Collecting Statistics (Part 2)

<strong>Your task:</strong> Now that your traffic and SYS flood are running, perform the following scenario:
- Ensure that your SYN cookies are remained off.
- Start legitimate traffic
- After 30 seconds start the attack
- After 120 seconds stop the attack
- After 30 seconds stop the legitimate traffic
- Stop the `tcpdump` on the `client`. Inside of your home (`/home/USERNAME_GOES_HERE`) directory on `client`, save the file as `tcpdump_cookies_off.txt`.

Use the text fields below to save your work. They will be used to generate a graph, so all fields will be required for Step 7 to work.

### Step 6: Collecting Statistics (Part 3)

<strong>Your task:</strong> Now, turn on the SYN cookies and repeat the steps above. Refer to Step 2 if you forgot how to enable SYN cookies.

After completing the steps, navigate to your home (`/home/USERNAME_GOES_HERE`) directory on `client`, then save the file as `tcpdump_cookies_on.txt`.

Use the text fields below to save your work. They will be used to generate a graph, so all fields will be required for Step 7 to work.

### Step 7: Generating a Graph

With the information that you gathered in the previous two steps, a graph of the data will be generated for you. Make sure that you save these graphs to include with your final submission.

<strong>Your task:</strong> Ensure that all text fields in Steps 5 and 6 are complete before continuing. Click on the button to generate a graph that shows your two results with a legend.

Once the graph has been generated, interpret the output and calculate connection duration for each TCP connection seen in the files. <strong>This will be a solution to the final step of the lab.</strong> Connection duration is the difference between the time of the first SYN and of the ACK following a FIN-ACK (or between the first SYN and the first RESET) on a connection. Recall what uniquely identifies a TCP connection, i.e. how to detect packets that belong to the same connection? If a connection did not end with a FIN or a RST, assign to it the duration of 200s. 

Once you're done thinking about these results, continue to the next step. If your graph generates correctly, this answer will be marked as correct. However, make sure that your results are sufficient and make sense for manual grading.

### Extra Credit Opportunity:

There are two extra credit questions.

1. Remove spoofing from the attack. Repeat the exercise without SYN cookies and observe and explain the effect. What happens? Can you explain why this happens? For hints, run a `tcpdump` on the `server` node and look for traffic patterns.
2. Can you modify the attack so that it is effective without spoofing and how would you do this?

These questions can be answered in the next step.

### Step 8: Creating a Final Submission

<strong>Your task:</strong> In addition to this lab, you are required to create a Word document with the following items (label each section):

1. Explanation how the TCP SYN flood attack works.
2. Explanation how SYN cookies work to prevent denial-of-service effect from SYN flood attack.
3. Your legitimate `client` script.
4. Your attack command (for `flooder`).
5. The connecton duration graphs that was generated in Step 7.
6. Explanation what happens in each case. Is the attack effective? How can you tell this from the graphs?
7. Answers to extra credit questions, if any.

This step cannot be automatically graded. Make sure that you include this with your manual submission.

## <strong>Grading</strong>

To check your overall grade, click on the button below.

In [25]:
# Click the button below to check your overall grade.
steps_to_check = [step_1, step_2, step_3, step_4, step_5, step_6, step_7, step_8, step_9, step_10, step_11, step_12, step_13, step_14, step_15, step_16, step_17, step_18, step_19, step_20, step_21]   

# Function to calculate grade after refreshing the cell
def calculate_grade(b):
    # To not auto-save at each step.
    global runAllSteps
    runAllSteps = True

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML("<span>Testing all steps. Please wait.</span> \
            <span><img width='12px' height='12px' style='margin-left: 3px;' src='resources/loading.gif'></span>"))
    
    # Required for checking the boolean values.
    for func in steps_to_check:
        func()  # Call each function in order.

    # Assuming steps are updated above this cell in some way
    steps = [step1Complete, step2Complete, step3Complete, step4Complete, step5Complete, step6Complete, step7Complete, step8Complete, step9Complete, step10Complete, step11Complete, step12Complete, step13Complete, step14Complete, step15Complete, step16Complete, step17Complete, step18Complete, step19Complete, step20Complete, step21Complete]
    output = ""
    stepsCorrect = 0
    numOfSteps = len(steps)

    for i in range(numOfSteps):
        if steps[i]:
            stepsCorrect += 1
            output += "<div style='color: green;'>Step " + str(i + 1) + " is complete.</div>"
        else:
            output += "<div style='color: red;'>Step " + str(i + 1) + " is incomplete.</div>"

    output += "<div style='color: black;'>You have " + str(stepsCorrect) + " out of " + str(numOfSteps) + " steps completed.</div>"

    with gradeOutput:
        gradeOutput.clear_output()
        display(HTML(output))

    # Makes auto-saving work again.
    runAllSteps = False
    
# Create a button to refresh the cell and another to calculate grade.
grade_button = widgets.Button(description="Calculate Grade")

# Link buttons to functions.
grade_button.on_click(calculate_grade)

# Output area.
gradeOutput = widgets.Output()

# Display the buttons and output.
display(grade_button, gradeOutput)

Button(description='Calculate Grade', style=ButtonStyle())

Output()

### Stopping the Lab

Once you are done with the lab, click on the "Stop Lab" button below. <strong>This will delete your materialization, which will delete all of the lab's resources.</strong> Your progress is saved automatically in ```saves/``` within the sidebar of your XDC. You may load this lab in the future by clicking "Load Lab" at the top.

In [26]:
# Click the button below to stop the experiment.
def stoplab(button):
    stop_lab(labname, confirm, stop_output)

# Creating the button.
stopButton = widgets.Button(description="Stop Lab")

# Create a confirmation check.
confirm = widgets.Checkbox(
    value=False,
    description='Confirm',
    disabled=False,
    indent=False
)

# Creating an output area.
stop_output = widgets.Output()

# Run the command on click.
stopButton.on_click(stoplab)

# Display the output.
display(confirm, stopButton, stop_output)

Checkbox(value=False, description='Confirm', indent=False)

Button(description='Stop Lab', style=ButtonStyle())

Output()